
# 04 — Tanore & Manda Formal Leakage / Distance Audit

এই **একটি notebook** Tanore ও Manda—দুই study area-এর **original point sample shapefile** দিয়ে formal leakage/distance audit করবে।

Audit checks:
- Train ↔ Validation exact coordinate overlap
- Within-training / within-validation duplicate coordinates
- প্রতিটি validation point থেকে nearest training point distance
- ≤3 m, ≤6 m, ≤10 m, ≤30 m proximity
- Same-class nearest training distance
- `sample_id` / `field_id` overlap, যদি attribute থাকে
- Risk summary, CSV/Excel/TXT report, histogram ও ECDF

**Important:** classification-এর pixel-extraction CSV দিয়ে এই audit করা হবে না। Original point shapefile ব্যবহার করা হবে।


### GitHub execution note
This notebook preserves the publication analysis logic. Local absolute paths were replaced with the portable `BORO_PROJECT_ROOT` setting. Run Jupyter from the repository root or set that environment variable before execution. Generated figures and tables are written below `Outputs/`; licensed source imagery is not included.


In [ ]:

# CELL 1 — Imports, paths, settings
from pathlib import Path
import os, re, warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path(os.environ.get('BORO_PROJECT_ROOT', str(Path.cwd()))).expanduser().resolve()
DATA_ROOT = PROJECT_ROOT / 'Data'
OUTPUT_ROOT = PROJECT_ROOT / 'Outputs' / 'Q1_Extensions' / 'Formal_Leakage_Distance_Audit'
TABLE_DIR = OUTPUT_ROOT / 'tables'
FIGURE_DIR = OUTPUT_ROOT / 'figures'
for p in [OUTPUT_ROOT, TABLE_DIR, FIGURE_DIR]: p.mkdir(parents=True, exist_ok=True)

TARGET_CRS='EPSG:32645'
THRESHOLDS=[3.0,6.0,10.0,30.0]
EXACT_TOL=0.001   # 1 mm
AREAS=['Tanore','Manda']
CLASSES=['Rice','NonRice']
SPLITS=['Training','Validation']
print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATA_ROOT   :', DATA_ROOT)
print('OUTPUT_ROOT :', OUTPUT_ROOT)


In [ ]:

# CELL 2 — Auto-find original sample shapefiles

def norm(s): return re.sub(r'[^a-z0-9]+','',str(s).lower())

EXPECTED={
 ('Tanore','Rice','Training'):['Tanore_rice_training'],
 ('Tanore','Rice','Validation'):['Tanore_Rice_Validation'],
 ('Tanore','NonRice','Training'):['Tanore_NonRice_Training'],
 ('Tanore','NonRice','Validation'):['Tanore_NonRice_Validation'],
 ('Manda','Rice','Training'):['Manda_rice_training'],
 ('Manda','Rice','Validation'):['Manda_Rice_Validation'],
 ('Manda','NonRice','Training'):['Manda_NonRice_Training'],
 ('Manda','NonRice','Validation'):['Manda_NonRice_Validation'],
}

if not DATA_ROOT.exists():
    raise FileNotFoundError(f'Data folder not found: {DATA_ROOT}')
ALL_SHP=list(DATA_ROOT.rglob('*.shp'))
print('Shapefiles found:', len(ALL_SHP))

def score(path, area, cls, split):
    full=norm(str(path)); stem=norm(path.stem); parent=norm(path.parent.name)
    exp={norm(x) for x in EXPECTED[(area,cls,split)]}
    s=0
    if stem in exp: s+=100
    if parent in exp: s+=90
    if any(e in full for e in exp): s+=60
    if norm(area) in full: s+=10
    if norm(split) in full: s+=10
    if cls=='NonRice':
        if 'nonrice' in full: s+=20
    else:
        if 'rice' in full and 'nonrice' not in full: s+=20
    if any(bad in full for bad in ['boundary','aoi','road','mask']): s-=50
    return s

def locate(area,cls,split):
    ranked=sorted([(score(p,area,cls,split),p) for p in ALL_SHP], key=lambda z:z[0], reverse=True)
    if not ranked or ranked[0][0] < 40:
        txt='\n'.join(f'{s:>4}  {p}' for s,p in ranked[:10])
        raise FileNotFoundError(f'Cannot locate {area} | {cls} | {split}. Top candidates:\n{txt}')
    return ranked[0][1]

paths={}
rows=[]
for a in AREAS:
    paths[a]={}
    for c in CLASSES:
        paths[a][c]={}
        for s in SPLITS:
            p=locate(a,c,s); paths[a][c][s]=p
            rows.append({'area':a,'class':c,'split':s,'path':str(p)})
located_files=pd.DataFrame(rows)
display(located_files)
print('✅ All 8 sample shapefiles located.')


In [ ]:

# CELL 3 — Load point samples and project to EPSG:32645
FIELD_CANDS=['field_id','fieldid','field','plot_id','plotid','parcel_id','parcelid']
ID_CANDS=['sample_id','sampleid','point_id','pointid','id','fid']

def find_col(cols,cands):
    m={norm(c):c for c in cols}
    for x in cands:
        if norm(x) in m: return m[norm(x)]
    return None

def load_layer(path,area,cls,split):
    g=gpd.read_file(path)
    if g.empty: raise ValueError(f'Empty shapefile: {path}')
    if g.crs is None: raise ValueError(f'Missing CRS: {path}')
    g=g[g.geometry.notna() & ~g.geometry.is_empty].copy()
    bad=set(g.geometry.geom_type.unique())-{'Point','MultiPoint'}
    if bad: raise ValueError(f'{path} has non-point geometry types: {bad}')
    g=g.explode(index_parts=False,ignore_index=True).to_crs(TARGET_CRS)
    fc=find_col(g.columns,FIELD_CANDS); ic=find_col(g.columns,ID_CANDS)
    out=gpd.GeoDataFrame({
      'area':area,'class_name':cls,'class':1 if cls=='Rice' else 0,'split':split,
      'source_file':str(path),'source_row':np.arange(len(g)),
      'field_id': g[fc].astype(str).replace('nan',np.nan) if fc else np.nan,
      'source_sample_id': g[ic].astype(str).replace('nan',np.nan) if ic else np.nan,
    }, geometry=g.geometry.values, crs=TARGET_CRS)
    out['x_m']=out.geometry.x; out['y_m']=out.geometry.y
    out['audit_id']=[f'{area}_{cls}_{split}_{i+1:04d}' for i in range(len(out))]
    return out

layers={}
for a in AREAS:
    frames=[load_layer(paths[a][c][s],a,c,s) for c in CLASSES for s in SPLITS]
    layers[a]=gpd.GeoDataFrame(pd.concat(frames,ignore_index=True),geometry='geometry',crs=TARGET_CRS)

summary=[]
for a,g in layers.items():
    for c in CLASSES:
        for s in SPLITS:
            z=g[(g.class_name==c)&(g.split==s)]
            summary.append({'area':a,'class':c,'split':s,'n_points':len(z),'field_id_available':bool(z.field_id.notna().any()),'sample_id_available':bool(z.source_sample_id.notna().any())})
input_summary=pd.DataFrame(summary)
display(input_summary)
print('✅ Samples loaded and projected.')


In [ ]:

# CELL 4 — Exact duplicate-coordinate audit

def xy_key(g):
    return g.x_m.round(3).astype(str)+'_'+g.y_m.round(3).astype(str)

dup_summary=[]; dup_details=[]
for a,g in layers.items():
    w=g.copy(); w['xy_key']=xy_key(w)
    tr=w[w.split=='Training'].copy(); va=w[w.split=='Validation'].copy()
    shared=set(tr.xy_key)&set(va.xy_key)
    d=pd.concat([tr[tr.xy_key.isin(shared)],va[va.xy_key.isin(shared)]])
    if len(d): d=d.copy(); d['duplicate_type']='Train_Validation_Exact'; dup_details.append(d)
    tmask=tr.xy_key.duplicated(keep=False); vmask=va.xy_key.duplicated(keep=False)
    if tmask.any():
        d=tr[tmask].copy(); d['duplicate_type']='Within_Training'; dup_details.append(d)
    if vmask.any():
        d=va[vmask].copy(); d['duplicate_type']='Within_Validation'; dup_details.append(d)
    dup_summary.append({'area':a,'train_validation_shared_coordinates':len(shared),'training_points_in_duplicate_coordinates':int(tmask.sum()),'validation_points_in_duplicate_coordinates':int(vmask.sum())})

duplicate_summary=pd.DataFrame(dup_summary)
duplicate_details=pd.concat(dup_details,ignore_index=True) if dup_details else pd.DataFrame()
display(duplicate_summary)


In [ ]:

# CELL 5 — Nearest training distance for every validation point
distance_frames=[]; dsum=[]
for a,g in layers.items():
    tr=g[g.split=='Training'].reset_index(drop=True).copy()
    va=g[g.split=='Validation'].reset_index(drop=True).copy()
    tree=cKDTree(tr[['x_m','y_m']].to_numpy(float))
    dist,idx=tree.query(va[['x_m','y_m']].to_numpy(float),k=1)
    A=va[['area','class_name','class','audit_id','source_sample_id','field_id','x_m','y_m']].copy()
    A.columns=['area','validation_class_name','validation_class','validation_audit_id','validation_source_sample_id','validation_field_id','validation_x_m','validation_y_m']
    A['nearest_training_distance_m']=dist
    for col in ['audit_id','class_name','class','source_sample_id','field_id','x_m','y_m']:
        A['nearest_training_'+col]=tr.iloc[idx][col].to_numpy()
    A['nearest_training_same_class']=A.validation_class.eq(A.nearest_training_class)
    same=np.full(len(va),np.nan)
    for cls in [0,1]:
        t=tr[tr['class']==cls]; mask=va['class'].to_numpy()==cls
        if len(t) and mask.any():
            dt,_=cKDTree(t[['x_m','y_m']].to_numpy(float)).query(va.loc[mask,['x_m','y_m']].to_numpy(float),k=1)
            same[np.where(mask)[0]]=dt
    A['nearest_same_class_training_distance_m']=same
    A['exact_coordinate_overlap']=A.nearest_training_distance_m<=EXACT_TOL
    for th in THRESHOLDS: A[f'within_{int(th)}m_of_training']=A.nearest_training_distance_m<=th
    distance_frames.append(A)
    row={'area':a,'n_training':len(tr),'n_validation':len(va),'exact_overlap_n':int(A.exact_coordinate_overlap.sum()),
         'nearest_distance_min_m':float(A.nearest_training_distance_m.min()),'nearest_distance_median_m':float(A.nearest_training_distance_m.median()),
         'nearest_distance_mean_m':float(A.nearest_training_distance_m.mean()),'nearest_distance_p05_m':float(A.nearest_training_distance_m.quantile(.05)),
         'nearest_distance_p25_m':float(A.nearest_training_distance_m.quantile(.25))}
    for th in THRESHOLDS:
        n=int((A.nearest_training_distance_m<=th).sum()); row[f'within_{int(th)}m_n']=n; row[f'within_{int(th)}m_pct']=100*n/len(A)
    dsum.append(row)

distance_details=pd.concat(distance_frames,ignore_index=True)
distance_summary=pd.DataFrame(dsum)
display(distance_summary.round(3))


In [ ]:

# CELL 6 — Shared sample ID / field ID audit
attr=[]
for a,g in layers.items():
    tr=g[g.split=='Training']; va=g[g.split=='Validation']
    ti=set(tr.source_sample_id.dropna().astype(str)); vi=set(va.source_sample_id.dropna().astype(str))
    tf=set(tr.field_id.dropna().astype(str)); vf=set(va.field_id.dropna().astype(str))
    si=ti&vi if ti and vi else set(); sf=tf&vf if tf and vf else set()
    attr.append({'area':a,'sample_id_available_train':bool(ti),'sample_id_available_validation':bool(vi),'shared_sample_id_count':len(si),
                 'shared_sample_ids':'; '.join(sorted(si)),'field_id_available_train':bool(tf),'field_id_available_validation':bool(vf),
                 'shared_field_id_count':len(sf),'shared_field_ids':'; '.join(sorted(sf)),'field_level_leakage_fully_assessable':bool(tf and vf)})
attribute_audit=pd.DataFrame(attr)
display(attribute_audit)


In [ ]:

# CELL 7 — Risk classification

def risk(a):
    d=distance_summary[distance_summary.area==a].iloc[0]; u=duplicate_summary[duplicate_summary.area==a].iloc[0]; x=attribute_audit[attribute_audit.area==a].iloc[0]
    reasons=[]
    if u.train_validation_shared_coordinates>0 or d.exact_overlap_n>0 or x.shared_sample_id_count>0 or x.shared_field_id_count>0:
        level='CRITICAL'; reasons.append('Exact train-validation overlap and/or shared sample/field ID detected.')
    elif d.within_6m_n>0:
        level='HIGH'; reasons.append('At least one validation point is within 6 m of a training point.')
    elif d.within_10m_n>0:
        level='MODERATE'; reasons.append('No ≤6 m overlap, but at least one validation point is within 10 m.')
    elif d.within_30m_n>0:
        level='CAUTION'; reasons.append('No ≤10 m overlap, but some validation points are within 30 m.')
    else:
        level='LOW'; reasons.append('No validation point is within 30 m of the nearest training point.')
    if not bool(x.field_level_leakage_fully_assessable): reasons.append('Field IDs unavailable in both splits; same-field leakage cannot be fully ruled out from point distance alone.')
    return {'area':a,'risk_level':level,'interpretation':' '.join(reasons)}
risk_summary=pd.DataFrame([risk(a) for a in AREAS])
display(risk_summary)


In [ ]:

# CELL 8 — Figures
for a in AREAS:
    d=distance_details[distance_details.area==a].nearest_training_distance_m.dropna().to_numpy()
    fig,ax=plt.subplots(figsize=(8,5)); ax.hist(d,bins=min(30,max(8,int(np.sqrt(len(d))))))
    for th in THRESHOLDS: ax.axvline(th,linestyle='--',linewidth=1)
    ax.set_xlabel('Nearest training-point distance (m)'); ax.set_ylabel('Validation-point count'); ax.set_title(f'{a}: Training–Validation Nearest-Distance Distribution')
    fig.tight_layout(); fig.savefig(FIGURE_DIR/f'{a}_Nearest_Training_Distance_Histogram.png',dpi=300,bbox_inches='tight'); plt.close(fig)

    ds=np.sort(d); y=np.arange(1,len(ds)+1)/len(ds)
    fig,ax=plt.subplots(figsize=(8,5)); ax.plot(ds,y,linewidth=1.8)
    for th in THRESHOLDS: ax.axvline(th,linestyle='--',linewidth=1)
    ax.set_xlabel('Nearest training-point distance (m)'); ax.set_ylabel('Cumulative fraction of validation points'); ax.set_ylim(0,1.01); ax.set_title(f'{a}: ECDF of Training–Validation Distance')
    fig.tight_layout(); fig.savefig(FIGURE_DIR/f'{a}_Nearest_Training_Distance_ECDF.png',dpi=300,bbox_inches='tight'); plt.close(fig)
print('✅ Figures saved.')


In [ ]:

# CELL 9 — Save CSV / Excel / TXT outputs
located_files.to_csv(TABLE_DIR/'Audit_Located_Sample_Files.csv',index=False)
input_summary.to_csv(TABLE_DIR/'Audit_Input_Summary.csv',index=False)
duplicate_summary.to_csv(TABLE_DIR/'Audit_Duplicate_Summary.csv',index=False)
if len(duplicate_details): duplicate_details.drop(columns=['geometry'],errors='ignore').to_csv(TABLE_DIR/'Audit_Duplicate_Details.csv',index=False)
distance_summary.to_csv(TABLE_DIR/'Audit_Distance_Threshold_Summary.csv',index=False)
distance_details.to_csv(TABLE_DIR/'Audit_Validation_Nearest_Training_Distances.csv',index=False)
attribute_audit.to_csv(TABLE_DIR/'Audit_FieldID_SampleID_Check.csv',index=False)
risk_summary.to_csv(TABLE_DIR/'Audit_Risk_Summary.csv',index=False)

excel_path=OUTPUT_ROOT/'Formal_Leakage_Distance_Audit_Results.xlsx'
with pd.ExcelWriter(excel_path,engine='openpyxl') as w:
    located_files.to_excel(w,sheet_name='Located_Files',index=False)
    input_summary.to_excel(w,sheet_name='Input_Summary',index=False)
    duplicate_summary.to_excel(w,sheet_name='Duplicates',index=False)
    distance_summary.to_excel(w,sheet_name='Distance_Summary',index=False)
    distance_details.to_excel(w,sheet_name='Validation_Distances',index=False)
    attribute_audit.to_excel(w,sheet_name='ID_Field_Audit',index=False)
    risk_summary.to_excel(w,sheet_name='Risk_Summary',index=False)

lines=['FORMAL TRAINING–VALIDATION LEAKAGE / DISTANCE AUDIT','='*68,f'Audit CRS: {TARGET_CRS}','Thresholds: 3 m, 6 m, 10 m, 30 m','']
for a in AREAS:
    d=distance_summary[distance_summary.area==a].iloc[0]; x=attribute_audit[attribute_audit.area==a].iloc[0]; r=risk_summary[risk_summary.area==a].iloc[0]
    lines += [a.upper(),'-'*68,f"Training points: {int(d.n_training)}",f"Validation points: {int(d.n_validation)}",f"Exact train-validation coordinate overlap: {int(d.exact_overlap_n)}"]
    for th in THRESHOLDS: lines.append(f"Validation points within {int(th)} m: {int(d[f'within_{int(th)}m_n'])} ({float(d[f'within_{int(th)}m_pct']):.2f}%)")
    lines += [f"Minimum nearest distance: {d.nearest_distance_min_m:.3f} m",f"Median nearest distance: {d.nearest_distance_median_m:.3f} m",f"Shared sample IDs: {int(x.shared_sample_id_count)}",f"Shared field IDs: {int(x.shared_field_id_count)}",f"Field-level leakage fully assessable: {bool(x.field_level_leakage_fully_assessable)}",f"Risk level: {r.risk_level}",f"Interpretation: {r.interpretation}",'']
lines += ['PUBLICATION NOTE','-'*68,'Point-distance checks diagnose spatial dependence but do not by themselves prove field-level independence. If field IDs/polygons are absent, state that limitation explicitly.']
summary_path=OUTPUT_ROOT/'Formal_Leakage_Distance_Audit_Summary.txt'
summary_path.write_text('\n'.join(lines),encoding='utf-8')

print('\n'+'='*78)
print('✅ FORMAL LEAKAGE / DISTANCE AUDIT COMPLETE')
print('='*78)
print('Excel  :',excel_path)
print('Summary:',summary_path)
print('Tables :',TABLE_DIR)
print('Figures:',FIGURE_DIR)



## কীভাবে result বুঝবেন

প্রথমে দেখবেন `Audit_Risk_Summary.csv`, তারপর `Audit_Distance_Threshold_Summary.csv`।

- **Exact overlap > 0** → critical leakage; sample review দরকার।
- **≤6 m > 0** → high spatial-dependence risk; বিশেষ করে 6 m point-buffer workflow হলে review দরকার।
- **≤10 m > 0** → moderate caution.
- **≤30 m > 0** → spatial autocorrelation caution; এটা নিজে থেকে leakage প্রমাণ করে না।
- `field_id` না থাকলে same-field independence পুরোপুরি verify করা যাবে না।

Notebook **কোনো sample delete করবে না**। শুধু audit ও suspicious points list করবে।
